In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
# compute groovist metrics (per-story profiles)
# for each story: groovist = tanh(groovist_raw)
# then aggregate across stories per model/prompt

MODELS = ['human', 'claude45', 'gpt4o', 'internvl3', 'llama4scout', 'qwen3vl']
PROMPTS = ['original', 'large']
GROOVIST_DIR = Path('/mimer/NOBACKUP/groups/naiss2024-6-297/bottom-up-anderson-vwp-features/models_scores_60')

In [3]:
# load groovist data for all model/prompt conditions
# original: one file per model
# large: seed42 for models, average seed42/43/44 for human

def load_groovist_scores(filepath):
    with open(filepath, 'r') as f:
        scores = json.load(f)
    return {int(sid): float(score) for sid, score in scores.items()}

def load_groovist_data():
    all_rows = []
    for model in MODELS:
        for prompt in PROMPTS:
            if prompt == 'original':
                gv_path = GROOVIST_DIR / f'scores_{model}_original.json'
                if not gv_path.exists():
                    print(f'Warning: missing {gv_path.name}')
                    continue
                gv_scores = load_groovist_scores(gv_path)
            else:
                if model == 'human':
                    seed_scores = {}
                    for seed in [42, 43, 44]:
                        gv_path = GROOVIST_DIR / f'scores_{model}_large_seed{seed}.json'
                        if gv_path.exists():
                            seed_scores[seed] = load_groovist_scores(gv_path)
                    if len(seed_scores) != 3:
                        print(f'Warning: missing one or more human large seeds for {model}_{prompt}')
                        continue
                    common_ids = set(seed_scores[42]) & set(seed_scores[43]) & set(seed_scores[44])
                    gv_scores = {sid: float(np.mean([seed_scores[s][sid] for s in [42, 43, 44]])) for sid in common_ids}
                else:
                    gv_path = GROOVIST_DIR / f'scores_{model}_large_seed42.json'
                    if not gv_path.exists():
                        print(f'Warning: missing {gv_path.name}')
                        continue
                    gv_scores = load_groovist_scores(gv_path)

            for story_id, gv_raw in gv_scores.items():
                all_rows.append({
                    'model': model,
                    'prompt': prompt,
                    'story_id': int(story_id),
                    'groovist_raw': float(gv_raw),
                    'groovist': float(np.tanh(gv_raw))
                })

    return pd.DataFrame(all_rows)

In [5]:
df_gv = load_groovist_data()
df_gv.head()

,model,prompt,story_id,groovist_raw,groovist
0,human,original,13683,0.454411,0.425518
1,human,original,13596,0.246281,0.241420
2,human,original,12423,0.257632,0.252079
3,human,original,12358,0.668782,0.584178
4,human,original,11719,0.188943,0.186727


In [6]:
gv_agg = (
    df_gv.groupby(['model', 'prompt'])
    .agg(
        groovist_raw_mean=('groovist_raw', 'mean'),
        groovist_mean=('groovist', 'mean'),
        groovist_std=('groovist', 'std'),
        observed_rows=('groovist', 'count'),
        total_rows=('groovist', 'size')
    )
    .reset_index()
    .assign(unit='story', missing_policy='all')
)
gv_agg

,model,prompt,groovist_raw_mean,groovist_mean,groovist_std,observed_rows,total_rows,unit,missing_policy
0,claude45,large,0.980687,0.732562,0.114427,60,60,story,all
1,claude45,original,1.018387,0.748818,0.109853,60,60,story,all
2,gpt4o,large,0.663911,0.562436,0.148620,60,60,story,all
3,gpt4o,original,0.659753,0.559174,0.148213,60,60,story,all
4,human,large,0.713500,0.584672,0.173493,60,60,story,all
5,human,original,0.574793,0.454289,0.286670,60,60,story,all
6,internvl3,large,0.703029,0.584757,0.150826,60,60,story,all
7,internvl3,original,0.722310,0.595736,0.155651,60,60,story,all
8,llama4scout,large,0.349255,0.310965,0.232497,60,60,story,all
9,llama4scout,original,0.409043,0.352517,0.249206,60,60,story,all


In [7]:

out_dir = Path('./analysis_data/groovist')
out_dir.mkdir(parents=True, exist_ok=True)

df_gv.to_csv(out_dir / 'groovist_metrics_raw.csv', index=False)
df_gv.to_csv(out_dir / 'groovist_metrics_per_story.csv', index=False)
gv_agg.to_csv(out_dir / 'groovist_metrics_agg.csv', index=False)
print('Saved GrooVist outputs to analysis_data/groovist/')

Saved GrooVist outputs to analysis_data/groovist/
